# Lab Day 10: Qwen3-VL-4B — định vị hư hại mặt đường
Notebook hoàn chỉnh E0–E4 theo `GUIDE.md`. Chạy từ trên xuống trên **Colab GPU T4 16 GB**.
E1–E3 dùng 300 optimizer steps mỗi lần; E4 chọn ablation kích thước dữ liệu 10%/100%.
Các cell chỉ định nghĩa hàm sẽ chưa huấn luyện cho tới phần 7. Không sửa công cụ chấm của đề.
Kết quả F1 và báo cáo phải lấy từ lần chạy thật; notebook không điền số liệu giả.

## 0. Môi trường (Colab T4)

In [ ]:
from pathlib import Path
import os, sys, subprocess

# Chạy được cả khi notebook đang ở repo local và khi mở riêng bằng Colab.
REPO_URL = "https://github.com/Le-Anh-Duy/K4-Track-4-Day10-VisionLanguageModel.git"
if not Path("requirements-colab.txt").is_file():
    checkout = Path("/content/K4-Track-4-Day10-VisionLanguageModel")
    if not checkout.exists():
        subprocess.run(["git", "clone", REPO_URL, str(checkout)], check=True)
    os.chdir(checkout)
ROOT = Path.cwd()
print("Repo:", ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                "requirements-colab.txt", "matplotlib", "pytest", "tqdm"], check=True)
# Nếu đã import torch/transformers trước khi cài: restart runtime rồi chạy lại.

In [ ]:
import gc, json, math, random, re, shutil, time
import torch
import yaml
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
from scripts.labcommon import CLASSES, read_jsonl

print("torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Cần CUDA GPU: Colab > Runtime > Change runtime type > T4 GPU.")
print(torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

# Bật để lưu checkpoint thẳng lên Drive; không bắt buộc tài khoản Drive khi chạy local.
USE_DRIVE = True
try:
    from google.colab import drive
except ImportError:
    drive = None
if USE_DRIVE and drive is not None:
    drive.mount("/content/drive")
    RUN_ROOT = Path("/content/drive/MyDrive/lab_day10/runs")
else:
    RUN_ROOT = ROOT / "work/runs"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT = ROOT / "data"

## 1. Dữ liệu: tải và chia split
Mặc định tải Kaggle. Có thể đổi `DATA_SOURCE` sang thư mục đã giải nén hoặc file zip.
Secrets chỉ được đọc khi cần tải; không ghi khóa vào notebook.

In [ ]:
DATA_SOURCE = "kaggle"  # hoặc "/content/dataset.zip", "/content/road-damage"
if not all((DATA_ROOT / "splits" / f"{s}.jsonl").is_file() for s in ("train", "val", "test")):
    args = [sys.executable, "scripts/prepare_data.py"]
    if DATA_SOURCE == "kaggle":
        if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
            try:
                from google.colab import userdata
                for key in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
                    os.environ[key] = userdata.get(key)
            except Exception as exc:
                raise RuntimeError("Cấu hình Colab Secrets KAGGLE_USERNAME/KAGGLE_KEY hoặc đổi DATA_SOURCE.") from exc
        args += ["--kaggle"]
    else:
        args += ["--zip" if DATA_SOURCE.lower().endswith(".zip") else "--source", DATA_SOURCE]
    subprocess.run(args, check=True)

splits = {s: read_jsonl(DATA_ROOT / "splits" / f"{s}.jsonl") for s in ("train", "val", "test")}
for s, rows in splits.items():
    assert rows, f"Split rỗng: {s}"
    counts = {c: sum(o["label"] == c for r in rows for o in r["objects"]) for c in CLASSES}
    print(s, len(rows), "images; boxes:", counts)
sessions = {s: {r["session"] for r in rows} for s, rows in splits.items()}
assert not (sessions["train"] & sessions["val"] or sessions["train"] & sessions["test"] or sessions["val"] & sessions["test"])
train_samples, val_samples, test_samples = (splits[s] for s in ("train", "val", "test"))

def open_image(sample):
    with Image.open(DATA_ROOT / sample["image_path"]) as image:
        return image.convert("RGB")  # giữ kích thước gốc, không thumbnail 512px

def draw_sample(sample, boxes=None):
    image = open_image(sample)
    draw = ImageDraw.Draw(image)
    colors = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}
    if boxes is None:
        boxes = []
        for obj in sample["objects"]:
            xc, yc, w, h = obj["yolo"]
            boxes.append({"label": obj["label"], "bbox_2d":
                          [(xc-w/2)*1000, (yc-h/2)*1000, (xc+w/2)*1000, (yc+h/2)*1000]})
    for obj in boxes:
        x1, y1, x2, y2 = obj["bbox_2d"]
        xy = [x1*image.width/1000, y1*image.height/1000, x2*image.width/1000, y2*image.height/1000]
        draw.rectangle(xy, outline=colors[obj["label"]], width=2)
        draw.text((xy[0], max(0, xy[1]-12)), obj["label"], fill=colors[obj["label"]])
    return image

fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, sample in zip(axes.flat, train_samples[:6]):
    ax.imshow(draw_sample(sample))
    ax.set_title(f"{sample['image_id']} ({len(sample['objects'])} boxes)", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
def yolo_to_box(label, yolo):
    if label not in CLASSES or len(yolo) != 4:
        raise ValueError("Nhãn/box không hợp lệ")
    if not all(math.isfinite(v) and 0 <= v <= 1 for v in yolo):
        raise ValueError("YOLO phải hữu hạn trong [0, 1]")
    xc, yc, w, h = yolo
    if w <= 0 or h <= 0:
        raise ValueError("Box phải có diện tích dương")
    xy = [max(0, min(1000, round(v*1000))) for v in (xc-w/2, yc-h/2, xc+w/2, yc+h/2)]
    # Giữ box ít nhất 1 đơn vị kể cả khi làm tròn tại biên 1000.
    for lo, hi in ((0, 2), (1, 3)):
        if xy[hi] <= xy[lo]:
            xy[lo] = min(xy[lo], 999)
            xy[hi] = xy[lo] + 1
    return {"label": label, "bbox_2d": xy}

def objects_to_answer(objects):
    boxes = [yolo_to_box(o["label"], o["yolo"]) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

PROMPT = ('Locate road damage: pothole, crack, manhole. Return ONLY a JSON list '
          'of {"label":"pothole|crack|manhole","bbox_2d":[x1,y1,x2,y2]}. '
          'Coordinates are integers normalized to 0-1000 relative to image width/height. '
          'Use top-left and bottom-right corners. If no objects, return [].')
assert objects_to_answer([]) == "[]"
assert yolo_to_box("crack", [0.5, 0.5, 0.2, 0.4])["bbox_2d"] == [400, 300, 600, 700]
assert yolo_to_box("crack", [1, 1, 0.0001, 0.0001])["bbox_2d"] == [999, 999, 1000, 1000]
sample = train_samples[0]
answer = objects_to_answer(sample["objects"])
for box in json.loads(answer):
    assert all(0 <= v <= 1000 for v in box["bbox_2d"])
print(answer)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, image, title in zip(axes, [draw_sample(sample), draw_sample(sample, json.loads(answer))], ["YOLO gốc", "JSON → pixel"]):
    ax.imshow(image); ax.set_title(title); ax.axis("off")
plt.show()

## 3. E0: zero-shot
NF4 chỉ cho LLM, giữ vision/projector và lm_head fp16; đặt toàn model trên GPU 0.
Giữ ảnh gốc, dùng chung processor/prompt cho mọi thí nghiệm. Output đúng `{image_id, raw}`.

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration, set_seed

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"

def load_base_model():
    quantization = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True, llm_int8_skip_modules=["visual", "lm_head"])
    return Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=quantization, dtype=torch.float16,
        device_map={"": 0}, attn_implementation="sdpa")

def user_message(image):
    return {"role": "user", "content": [{"type": "image", "image": image},
                                          {"type": "text", "text": PROMPT}]}

def predict(model, rows, path, batch_size=4, max_new_tokens=256):
    model.eval()
    model.config.use_cache = True
    processor.tokenizer.padding_side = "left"
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # Ghi file tạm: lần chạy bị ngắt không thay thế predictions hoàn chỉnh cũ.
    temporary = path.with_suffix(".jsonl.tmp")
    with temporary.open("w", encoding="utf-8") as handle, torch.inference_mode():
        for start in tqdm(range(0, len(rows), batch_size), desc=path.stem):
            chunk = rows[start:start+batch_size]
            chats = [[user_message(open_image(row))] for row in chunk]
            inputs = processor.apply_chat_template(chats, tokenize=True, add_generation_prompt=True,
                       return_dict=True, return_tensors="pt", padding=True)
            inputs.pop("token_type_ids", None)
            inputs = inputs.to(model.device)
            with torch.autocast("cuda", dtype=torch.float16):
                generated = model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens)
            raw = processor.batch_decode(generated[:, inputs["input_ids"].shape[1]:],
                      skip_special_tokens=True, clean_up_tokenization_spaces=False)
            for row, text in zip(chunk, raw):
                handle.write(json.dumps({"image_id": row["image_id"], "raw": text}, ensure_ascii=False)+"\n")
            del inputs, generated
    temporary.replace(path)
    return path

def score_run(pred_path, name, split="test", meta=None, evidence_name=None):
    out = ROOT / "report/evidence" / f"{evidence_name or name.lower()}.json"
    cmd = [sys.executable, "scripts/score.py", "--pred", str(pred_path), "--name", name,
           "--split", split, "--out", str(out)]
    for key, value in (meta or {}).items():
        cmd += ["--meta", f"{key}={value}"]
    subprocess.run(cmd, check=True)
    return json.loads(out.read_text(encoding="utf-8"))

model = load_base_model()
predict(model, test_samples, ROOT / "work/e0.jsonl")
e0 = score_run(ROOT / "work/e0.jsonl", "E0")
for row in read_jsonl(ROOT / "work/e0.jsonl")[:5]:
    print(row)

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss
Encode prompt và hội thoại đầy đủ bằng **cùng processor và ảnh**; xác nhận prefix trước khi mask.
Chỉ tính loss trên JSON và token kết thúc. Pad các tensor chuỗi bên phải; nối patch ảnh/grid
theo chiều 0, không pad patch ảnh. Không cắt chuỗi vì có thể mất token ảnh hoặc JSON.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

def encode_example(image, answer):
    user = user_message(image)
    kwargs = dict(tokenize=True, return_dict=True, return_tensors="pt")
    prompt = processor.apply_chat_template([user], add_generation_prompt=True, **kwargs)
    full = processor.apply_chat_template([user, {"role": "assistant", "content": answer}],
                                          add_generation_prompt=False, **kwargs)
    full.pop("token_type_ids", None)
    n = prompt["input_ids"].shape[1]
    if not torch.equal(full["input_ids"][0, :n], prompt["input_ids"][0]):
        raise ValueError("Chat template thay đổi prefix: không được mask bằng độ dài ước lượng.")
    labels = full["input_ids"].clone()
    labels[:, :n] = -100
    labels[full["attention_mask"] == 0] = -100
    assert (labels != -100).any(), "Không còn token đáp án"
    full["labels"] = labels
    return dict(full)

def collate(batch):
    result = {}
    sequence_keys = {"input_ids", "attention_mask", "labels", "mm_token_type_ids"}
    if any(set(item) != set(batch[0]) for item in batch):
        raise ValueError("Batch có bộ trường không đồng nhất")
    for key in batch[0]:
        values = [item[key] for item in batch]
        if key in sequence_keys:
            pad = -100 if key == "labels" else (processor.tokenizer.pad_token_id if key == "input_ids" else 0)
            result[key] = pad_sequence([v.squeeze(0) for v in values], batch_first=True, padding_value=pad)
        elif key in {"pixel_values", "image_grid_thw"}:
            result[key] = torch.cat(values, dim=0)
        else:
            raise ValueError(f"Trường processor chưa xử lý: {key}")
    return result

class RoadDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        return encode_example(open_image(row), objects_to_answer(row["objects"]))

encoded = RoadDataset(train_samples)[0]
supervised = encoded["labels"][0] != -100
decoded = processor.tokenizer.decode(encoded["input_ids"][0][supervised], skip_special_tokens=False)
expected = objects_to_answer(train_samples[0]["objects"])
assert decoded.startswith(expected) and "<|im_end|>" in decoded[len(expected):]
image_id = model.config.image_token_id
image_tokens = int((encoded["input_ids"] == image_id).sum())
grid_tokens = int(encoded["image_grid_thw"].prod(-1).sum()) // processor.image_processor.merge_size**2
assert image_tokens == grid_tokens
print("Vision tokens:", image_tokens, "(640×360 mặc định của đề: 220)")
print("Chỉ các token này có loss:", decoded)
assert (encoded["labels"][encoded["input_ids"] == image_id] == -100).all()
batch_check = collate([encoded, encode_example(open_image(train_samples[1]), "[]")])
assert (batch_check["labels"][batch_check["attention_mask"] == 0] == -100).all()
assert batch_check["image_grid_thw"].shape[0] == 2
assert batch_check["pixel_values"].shape[0] == sum(int(g.prod()) for g in batch_check["image_grid_thw"])
del encoded, batch_check
# Giải phóng E0 trước khi tạo model mới cho từng thí nghiệm.
del model
gc.collect()
torch.cuda.empty_cache()

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel

LLM_TARGET = r"model\.language_model\.layers\.\d+\.(?:self_attn\.(?:q|k|v|o)_proj|mlp\.(?:gate|up|down)_proj)"
VIT_TARGET = r"model\.visual\.blocks\.\d+\.(?:attn\.(?:qkv|proj)|mlp\.(?:linear_fc1|linear_fc2))"

def is_projector(name):
    return "visual.merger." in name or "visual.deepstack_merger_list." in name

def apply_setup(base, cfg):
    base.requires_grad_(False)
    regex = LLM_TARGET if cfg["lora"]["targets"] == "llm" else f"(?:{LLM_TARGET}|{VIT_TARGET})"
    matched = [n for n, _ in base.named_modules() if re.fullmatch(regex, n)]
    expected = 252 if cfg["lora"]["targets"] == "llm" else 348
    if len(matched) != expected:
        raise ValueError(f"Target LoRA: {len(matched)} != {expected}; kiểm tra named_modules/model version.")
    # Không dùng prepare_model_for_kbit_training: nó cast toàn bộ ViT sang fp32,
    # tăng VRAM ngoài yêu cầu; chỉ projector cần fp32 cho GradScaler.
    tuned = get_peft_model(base, LoraConfig(
        r=cfg["lora"]["r"], lora_alpha=cfg["lora"]["alpha"],
        lora_dropout=cfg["lora"]["dropout"], target_modules=regex, bias="none"))
    for name, param in tuned.named_parameters():
        if is_projector(name) and cfg["train_projector"]:
            param.data = param.data.float()
            param.requires_grad_(True)
        elif param.requires_grad:
            param.data = param.data.float()  # fp16 gradients không unscale được
    tuned.enable_input_require_grads()
    tuned.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    tuned.config.use_cache = False
    assert all("lora_" in n or (cfg["train_projector"] and is_projector(n))
               for n, p in tuned.named_parameters() if p.requires_grad)
    assert all(p.dtype == torch.float32 for p in tuned.parameters() if p.requires_grad)
    tuned.print_trainable_parameters()
    return tuned

## 6. Vòng huấn luyện
AdamW + warmup/cosine, AMP fp16, tích lũy gradient, checkpoint mỗi 100 optimizer steps.
Lưu adapter **và projector**; nạp lại từ ổ đĩa khi suy luận để kiểm tra checkpoint.
Checkpoint dùng để suy luận; nếu runtime ngắt giữa một run, chạy lại run đó từ đầu
(không giả vờ resume optimizer). Có thể giảm batch=1, accum=8 đồng đều mọi setup nếu OOM.

In [ ]:
from transformers import get_cosine_schedule_with_warmup

def save_checkpoint(model, folder, cfg, stats):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(folder)
    processor.save_pretrained(folder)
    extra = {n: p.detach().cpu().clone() for n, p in model.named_parameters()
             if p.requires_grad and is_projector(n)}
    if cfg["train_projector"]:
        assert extra, "Thiếu projector cần lưu"
        torch.save(extra, folder / "extra_trainable.pt")
    (folder / "run.json").write_text(json.dumps({**stats, "config": cfg}, indent=2), encoding="utf-8")

def train_run(cfg, rows, folder):
    set_seed(cfg["seed"])
    model = apply_setup(load_base_model(), cfg)
    params = [p for p in model.parameters() if p.requires_grad]
    count = sum(p.numel() for p in params)
    loader = DataLoader(RoadDataset(rows), batch_size=cfg["batch_size"], shuffle=True,
                        collate_fn=collate, num_workers=0)
    iterator = iter(loader)
    optimizer = torch.optim.AdamW(params, lr=float(cfg["lr"]), weight_decay=0)
    scheduler = get_cosine_schedule_with_warmup(optimizer,
        num_warmup_steps=max(1, math.ceil(cfg["max_steps"]*0.05)), num_training_steps=cfg["max_steps"])
    scaler = torch.amp.GradScaler("cuda", init_scale=128.0)
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    start = time.perf_counter()
    losses = []
    model.train()
    optimizer.zero_grad(set_to_none=True)
    for step in tqdm(range(1, cfg["max_steps"]+1), desc=cfg["name"]):
        loss_sum = 0.0
        for _ in range(cfg["grad_accum"]):
            try:
                batch = next(iterator)
            except StopIteration:
                iterator = iter(loader)
                batch = next(iterator)
            batch = {k: v.to(model.device) for k, v in batch.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(**batch).loss
            if not torch.isfinite(loss).item():
                raise FloatingPointError(f"{cfg['name']} step {step}: loss NaN/Inf; giảm lr/kiểm tra mask.")
            loss_sum += loss.detach().item()/cfg["grad_accum"]
            scaler.scale(loss/cfg["grad_accum"]).backward()
            del loss, batch
        scaler.unscale_(optimizer)
        norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
        if not torch.isfinite(norm).item():
            raise FloatingPointError(f"{cfg['name']} step {step}: gradient NaN/Inf; giảm lr hoặc AMP scale.")
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        losses.append(loss_sum)
        if step % 10 == 0:
            print(f"step={step} loss={loss_sum:.4f}")
        if step % 100 == 0 or step == cfg["max_steps"]:
            torch.cuda.synchronize()
            stats = dict(trainable_params=count, peak_vram_gb=torch.cuda.max_memory_allocated()/1024**3,
                         train_minutes=(time.perf_counter()-start)/60, steps=step, losses=losses)
            save_checkpoint(model, folder, cfg, stats)
    del model, optimizer, scheduler, scaler, params, loader, iterator
    gc.collect()
    torch.cuda.empty_cache()
    return stats

def load_checkpoint(folder):
    folder = Path(folder)
    run = json.loads((folder / "run.json").read_text(encoding="utf-8"))
    model = PeftModel.from_pretrained(load_base_model(), folder, is_trainable=False)
    if run["config"]["train_projector"]:
        extra = torch.load(folder / "extra_trainable.pt", map_location="cpu", weights_only=True)
        named = dict(model.named_parameters())
        expected = {n for n in named if is_projector(n)}
        if set(extra) != expected:
            raise ValueError("Projector checkpoint không khớp model")
        with torch.no_grad():
            for name, weight in extra.items():
                if named[name].shape != weight.shape:
                    raise ValueError(f"Sai shape projector: {name}")
                named[name].copy_(weight.to(device=named[name].device, dtype=named[name].dtype))
    return model, run

## 7. Huấn luyện, suy luận và chấm E1–E3
Mỗi setup bắt đầu từ model gốc. Chấm val để chọn setup cho E4; test để báo cáo.
`REUSE_COMPLETED=True` bỏ qua run đã đủ steps với cùng config và dataset version.
Để debug nhanh, sửa đồng đều `MAX_STEPS=2`; kết quả đó **không phải** run nộp bài 300 steps.

In [ ]:
MAX_STEPS = 300
BATCH_SIZE, GRAD_ACCUM = 2, 4
REUSE_COMPLETED = True
dataset_version = json.loads((DATA_ROOT / "splits/meta.json").read_text(encoding="utf-8"))["dataset_version"]

def run_experiment(cfg, rows):
    cfg = dict(cfg, dataset_version=dataset_version)
    folder = RUN_ROOT / cfg["name"].lower()
    run_path = folder / "run.json"
    old = json.loads(run_path.read_text(encoding="utf-8")) if run_path.exists() else None
    complete = old and old["config"] == cfg and old["steps"] == cfg["max_steps"]
    if not (REUSE_COMPLETED and complete):
        train_run(cfg, rows, folder)
    model, run = load_checkpoint(folder)
    meta = {k: run[k] for k in ("trainable_params", "peak_vram_gb", "train_minutes")}
    results = {}
    for split in ("val", "test"):
        tag = cfg["name"].lower() + ("_val" if split == "val" else "")
        prediction = ROOT / "work" / f"{tag}.jsonl"
        predict(model, splits[split], prediction)
        results[split] = score_run(prediction, cfg["name"], split=split, meta=meta, evidence_name=tag)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return results

configs, results = {}, {}
for number in (1, 2, 3):
    cfg = yaml.safe_load((ROOT / f"configs/e{number}.yaml").read_text(encoding="utf-8"))
    cfg.update(max_steps=MAX_STEPS, batch_size=BATCH_SIZE, grad_accum=GRAD_ACCUM)
    configs[cfg["name"]] = cfg
    results[cfg["name"]] = run_experiment(cfg, train_samples)

assert results["E2"]["test"]["meta"]["trainable_params"] > results["E1"]["test"]["meta"]["trainable_params"] + 100_000_000
for name, result in results.items():
    r = result["test"]
    print(name, "F1=", r["micro"]["f1"], "CI95=", r["f1_ci95"], "meta=", r["meta"])

## 8. E4: ablation kích thước dữ liệu (10% / 100%)
Chọn setup tốt nhất **theo val**, giữ nguyên rank/lr/steps/prompt, chỉ đổi lượng dữ liệu.
Mốc 100% dùng lại run tương ứng; 10% là tập con cố định theo seed. Vẫn giữ 300 steps,
nên so sánh đo lượng dữ liệu với ngân sách optimizer cố định, không cố định số epoch.

In [ ]:
import copy
best_name = max(results, key=lambda name: results[name]["val"]["micro"]["f1"])
best_cfg = copy.deepcopy(configs[best_name])
print("Best theo val:", best_name)
baseline = copy.deepcopy(results[best_name]["test"])
baseline.update(name="E4_data_size_1.0")
baseline["meta"].update(ablation="data_size", train_fraction=1.0, base_setup=best_name)
out = ROOT / "report/evidence/e4_data_size_1.0.json"
out.write_text(json.dumps(baseline, indent=2), encoding="utf-8")
subset = random.Random(best_cfg["seed"]).sample(train_samples, max(1, round(len(train_samples)*0.1)))
e4_cfg = copy.deepcopy(best_cfg)
e4_cfg.update(name="E4_data_size_0.1", train_fraction=0.1, base_setup=best_name)
results["E4_data_size_0.1"] = run_experiment(e4_cfg, subset)
print("100%:", baseline["micro"], "CI95:", baseline["f1_ci95"])
print("10%:", results["E4_data_size_0.1"]["test"]["micro"], "CI95:", results["E4_data_size_0.1"]["test"]["f1_ci95"])

### E5 (tuỳ chọn): reward IoU và self-check
Phần bắt buộc dừng ở E4. Hàm dưới thực hiện reward theo GUIDE; **chưa chạy GRPO**,
không sinh evidence E5. Muốn lấy bonus cần thêm GRPOTrainer và chạy/chấm GPU thật.

In [ ]:
from scripts.score import parse_output, iou_xyxy

def iou_reward(completion, gt_boxes):
    pred = parse_output(completion)
    if pred is None:
        return 0.0
    if not pred and not gt_boxes:
        return 1.0
    candidates = sorted(((iou_xyxy(g["bbox"], p["bbox"]), i, j)
                         for i, g in enumerate(gt_boxes) for j, p in enumerate(pred)
                         if g["label"] == p["label"]), reverse=True)
    used_gt, used_pred, total = set(), set(), 0.0
    for overlap, i, j in candidates:
        if overlap >= 0.5 and i not in used_gt and j not in used_pred:
            used_gt.add(i); used_pred.add(j); total += overlap
    return 0.2 + 0.8*total/max(len(gt_boxes), len(pred), 1)

perfect = '[{"label":"pothole","bbox_2d":[100,100,200,200]}]'
gt = parse_output(perfect)
assert iou_reward(perfect, gt) == 1.0
assert iou_reward("broken JSON", gt) == 0.0
assert iou_reward(perfect.replace("pothole", "crack"), gt) == 0.2
assert abs(iou_reward(json.dumps(json.loads(perfect)*2), gt) - 0.6) < 1e-9
assert iou_reward("[]", []) == 1.0
assert iou_reward("[]", gt) == 0.2
print("Reward self-checks passed")

### Tổng hợp và nộp bài
Điền `REPORT.md` dựa trên evidence thật và output đã xem; bổ sung ảnh minh họa, commit theo yêu cầu.
Không chạy `scripts/make_notebook.py`: script của đề sẽ ghi đè notebook đã hoàn thiện bằng khung trống.
Tham khảo API: [Qwen3-VL](https://huggingface.co/docs/transformers/model_doc/qwen3_vl),
[PEFT quantization](https://huggingface.co/docs/peft/developer_guides/quantization).

In [ ]:
for path in sorted((ROOT / "report/evidence").glob("*.json")):
    r = json.loads(path.read_text(encoding="utf-8"))
    print(f"{path.name:28s} F1={r['micro']['f1']:.4f} CI95={r['f1_ci95']} valid={r['parse_valid_rate']:.3f}")
# Chỉ bật sau khi điền REPORT.md và thêm ảnh minh họa/commit.
CHECK_SUBMISSION = False
if CHECK_SUBMISSION:
    subprocess.run([sys.executable, "scripts/check_submission.py"], check=True)